# LOAO Fold Generation (v2 -- length-matched negatives)

Generates Leave-One-Application-Out cross-validation folds for fine-tuning.

**Fix applied here (v1 had a bug):** v1 used full benign windows (mean 1071 tokens,
up to 5026) as negatives against 2-request attack exemplars (~20-60 tokens). The
classifier learned "short = attack, long = benign" instead of any real content
signal -- confirmed by 0.9998 vs 0.00007 probability separation on every fold/seed.

v2 explodes each benign window into consecutive 2-request sub-sequences, so every
negative has the same shape as every positive. Length can no longer be used as a
shortcut.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Prerequisites (upload before running)

- `{BASE}/data/tokenised/benign_windows.jsonl` (340 benign windows, from notebook 02)
- `{BASE}/data/labeled/track_a_attacks_final.jsonl` (10 exemplars)
- `{BASE}/data/labeled/track_b_attacks_final.jsonl` (22 exemplars)

In [2]:
"""
Generate LOAO folds with LENGTH-MATCHED negatives.

Fix for the v1 bug: v1 used full benign windows (mean 1071 tokens, up to 5026)
as negatives against 2-request attack exemplars (~20-60 tokens). The classifier
learned "short = attack, long = benign" instead of any real content signal --
confirmed by 0.9998 vs 0.00007 probability separation on every fold/seed.

v2 extracts consecutive 2-request sub-sequences from each benign window, so
every negative has the same shape (2 requests joined by " [SEP] ") as every
positive. Length can no longer be used as a shortcut.
"""

import json
import os
import random

random.seed(42)

BASE = "/content/drive/MyDrive/thesis"

CVE_TO_APP = {
    "CVE_2023_46449": "phpapp_ample", "CVE_2024_3139": "phpapp_lms",
    "CVE_2024_40480": "phpapp_quiz", "CVE_2024_55232": "phpapp_notes",
    "CVE_2024_7658": "phpapp_filemgmt", "CVE_2024_9082": "phpapp_oews",
    "CVE_2025_0802": "phpapp_leaveborrow", "CVE_2025_0843": "phpapp_studentmgmt",
    "CVE_2025_3537": "phpapp_leavemgmt2",
    "CVE_2023_36100": "IceCMS", "CVE_2024_1313": "Grafana",
    "CVE_2024_7041": "OpenWebUI", "CVE_2026_45671": "OpenWebUI",
    "Zammad_IDOR": "Zammad", "CVE_2026_55255": "Langflow",
    "CVE_2026_30843": "Wekan", "CVE_2026_1892": "Wekan", "CVE_2026_1894": "Wekan",
    "CVE_2026_25758": "Spree",
}

BENIGN_TAG_TO_APP = {
    "CVE_2023_46449": "phpapp_ample", "CVE_2024_3139": "phpapp_lms",
    "CVE_2024_40480": "phpapp_quiz", "CVE_2024_55232": "phpapp_notes",
    "CVE_2024_7658": "phpapp_filemgmt", "CVE_2024_9082": "phpapp_oews",
    "CVE_2025_0802": "phpapp_leaveborrow", "CVE_2025_3537": "phpapp_leavemgmt2",
    "Grafana_CVE_2024_1313": "Grafana", "IceCMS_CVE_2023_36100": "IceCMS",
    "Langflow_CVE_2026_55255": "Langflow", "OpenWebUI_CVE_2024_7041": "OpenWebUI",
    "OpenWebUI_CVE_2026_45671": "OpenWebUI", "Spree_CVE_2026_25757": "Spree",
    "Wekan_IDOR_CVE": "Wekan", "Zammad_IDOR_CVE": "Zammad",
}

NEG_RATIO = 4


def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]


def explode_to_pairs(benign_windows):
    """Extract every consecutive 2-request sub-sequence from each benign window.
    Each output pair inherits the parent window's app tag.""
    pairs = []
    for w in benign_windows:
        requests = w["sequence"].split(" [SEP] ")
        app = w["app"]
        if len(requests) < 2:
            continue  # can't form a pair, skip
        for i in range(len(requests) - 1):
            pair_seq = requests[i] + " [SEP] " + requests[i + 1]
            pairs.append({"sequence": pair_seq, "app": app})
    return pairs


def main():
    attacks = load_jsonl(f"{BASE}/data/labeled/track_a_attacks_final.jsonl") + \
              load_jsonl(f"{BASE}/data/labeled/track_b_attacks_final.jsonl")
    benign = load_jsonl(f"{BASE}/data/tokenised/benign_windows.jsonl")

    for a in attacks:
        a["app"] = CVE_TO_APP.get(a["cve"])
    attacks = [a for a in attacks if a["app"] is not None]
    apps = sorted(set(a["app"] for a in attacks))

    for b in benign:
        b["app"] = BENIGN_TAG_TO_APP.get(b.get("cve"), "OTHER")

    # KEY FIX: explode full windows into length-matched 2-request pairs
    benign_pairs = explode_to_pairs(benign)
    print(f"Exploded {len(benign)} benign windows into {len(benign_pairs)} "
          f"length-matched 2-request pairs\n")

    # Sanity check: report length distribution comparison
    attack_lengths = [len(a["sequence"].split()) for a in attacks]
    pair_lengths = [len(p["sequence"].split()) for p in benign_pairs]
    print(f"Attack exemplar lengths (words): min={min(attack_lengths)}, "
          f"max={max(attack_lengths)}, mean={sum(attack_lengths)/len(attack_lengths):.1f}")
    print(f"Benign pair lengths (words): min={min(pair_lengths)}, "
          f"max={max(pair_lengths)}, mean={sum(pair_lengths)/len(pair_lengths):.1f}\n")

    os.makedirs(f"{BASE}/data/loao_folds_v2", exist_ok=True)

    for held_out_app in apps:
        fold_dir = f"{BASE}/data/loao_folds_v2/fold_{held_out_app}"
        os.makedirs(fold_dir, exist_ok=True)

        train_attacks = [a for a in attacks if a["app"] != held_out_app]
        eval_attacks = [a for a in attacks if a["app"] == held_out_app]

        n_train_neg = min(len(benign_pairs), len(train_attacks) * NEG_RATIO)
        train_benign = random.sample(benign_pairs, n_train_neg)

        held_out_pairs = [p for p in benign_pairs if p["app"] == held_out_app]
        n_eval_neg = min(len(held_out_pairs), 10)
        eval_benign = random.sample(held_out_pairs, n_eval_neg) if held_out_pairs else []

        train_records = (
            [{"sequence": a["sequence"], "label": 1} for a in train_attacks] +
            [{"sequence": b["sequence"], "label": 0} for b in train_benign]
        )
        eval_records = (
            [{"sequence": a["sequence"], "label": 1} for a in eval_attacks] +
            [{"sequence": b["sequence"], "label": 0} for b in eval_benign]
        )
        random.shuffle(train_records)

        with open(f"{fold_dir}/train.jsonl", "w") as f:
            for r in train_records:
                f.write(json.dumps(r) + "\n")
        with open(f"{fold_dir}/eval.jsonl", "w") as f:
            for r in eval_records:
                f.write(json.dumps(r) + "\n")

        print(f"fold_{held_out_app}: train={len(train_records)} "
              f"({len(train_attacks)} pos / {len(train_benign)} neg), "
              f"eval={len(eval_records)} ({len(eval_attacks)} pos / {len(eval_benign)} neg)"
              + ("  <-- NO EVAL NEGATIVES" if not eval_benign else ""))


if __name__ == "__main__":
    main()


Exploded 340 benign windows into 15305 length-matched 2-request pairs

Attack exemplar lengths (words): min=7, max=9, mean=8.4
Benign pair lengths (words): min=7, max=9, mean=7.2

fold_Grafana: train=155 (31 pos / 124 neg), eval=11 (1 pos / 10 neg)
fold_IceCMS: train=155 (31 pos / 124 neg), eval=11 (1 pos / 10 neg)
fold_Langflow: train=155 (31 pos / 124 neg), eval=11 (1 pos / 10 neg)
fold_OpenWebUI: train=150 (30 pos / 120 neg), eval=12 (2 pos / 10 neg)
fold_Spree: train=155 (31 pos / 124 neg), eval=11 (1 pos / 10 neg)
fold_Wekan: train=145 (29 pos / 116 neg), eval=13 (3 pos / 10 neg)
fold_Zammad: train=155 (31 pos / 124 neg), eval=11 (1 pos / 10 neg)
fold_phpapp_ample: train=130 (26 pos / 104 neg), eval=16 (6 pos / 10 neg)
fold_phpapp_filemgmt: train=155 (31 pos / 124 neg), eval=11 (1 pos / 10 neg)
fold_phpapp_leaveborrow: train=145 (29 pos / 116 neg), eval=13 (3 pos / 10 neg)
fold_phpapp_leavemgmt2: train=150 (30 pos / 120 neg), eval=12 (2 pos / 10 neg)
fold_phpapp_lms: train=150 (30